In [1]:
# ============================================================
# TEST 1: NonKYC REST API — Balance format verification
# ============================================================
# This verifies the exact field names and types in /balances
# We need to confirm: available, held, and that available+held = total
# 
# REQUIRES: Your NonKYC API key and secret

import requests
import hmac
import hashlib
import time
import json
import os

NONKYC_API_KEY = os.environ["NONKYC_API_KEY"]
NONKYC_API_SECRET = os.environ["NONKYC_API_SECRET"]
NONKYC_BASE_URL = "https://api.nonkyc.io/api/v2"

MEXC_KEY = os.environ.get("MEXC_API_KEY", "")
MEXC_SECRET = os.environ.get("MEXC_API_SECRET", "")
MEXC_BASE = "https://api.mexc.com"


In [2]:
import requests, json
import numpy as np

NONKYC = "https://api.nonkyc.io/api/v2"

def probe(symbol, bands=(0.02, 0.05, 0.10, 0.25)):
    print("=" * 70); print(symbol)
    r = requests.get(f"{NONKYC}/market/orderbook", params={"symbol": symbol}, timeout=20)
    print("  HTTP", r.status_code)
    ob = r.json()
    print("  top-level keys:", list(ob)[:12])
    bids, asks = ob.get("bids", []), ob.get("asks", [])
    print(f"  levels returned: {len(bids)} bids / {len(asks)} asks")
    if bids:
        print("  raw bid[0]:", json.dumps(bids[0])[:200])
        print("  raw ask[0]:", json.dumps(asks[0])[:200])
    def lv(x):
        if isinstance(x, dict):
            return (float(x.get("price", x.get("p", 0))),
                    float(x.get("quantity", x.get("q", x.get("amount", 0)))))
        return float(x[0]), float(x[1])
    B = [lv(x) for x in bids]; A = [lv(x) for x in asks]
    if not B or not A:
        print("  EMPTY BOOK"); return
    bb, ba = max(p for p, _ in B), min(p for p, _ in A)
    mid = (bb + ba) / 2
    print(f"  best bid {bb} / best ask {ba} / mid {mid:.6f} / spread {(ba-bb)/mid*100:.3f}%")
    print(f"  TOTAL book notional: bids ${sum(p*q for p,q in B):,.0f}  "
          f"asks ${sum(p*q for p,q in A):,.0f}")
    for band in bands:
        lo, hi = mid * (1 - band), mid * (1 + band)
        d = sum(p*q for p, q in B if p >= lo) + sum(p*q for p, q in A if p <= hi)
        nb = sum(1 for p, _ in B if p >= lo); na = sum(1 for p, _ in A if p <= hi)
        print(f"  +-{band*100:>4.1f}% band: ${d:>12,.2f}   ({nb} bids, {na} asks in band)")
    print("  nearest 6 bids:", [(p, q) for p, q in sorted(B, reverse=True)[:6]])
    print("  nearest 6 asks:", [(p, q) for p, q in sorted(A)[:6]])

for s in ("DASH/USDT", "XMR/USDT", "BELLS/USDT"):
    probe(s)

# does the endpoint truncate without a limit param?
for lim in (None, 100, 500):
    p = {"symbol": "DASH/USDT"} | ({"limit": lim} if lim else {})
    ob = requests.get(f"{NONKYC}/market/orderbook", params=p, timeout=20).json()
    print(f"limit={lim}: {len(ob.get('bids', []))} bids / {len(ob.get('asks', []))} asks")

DASH/USDT
  HTTP 200
  top-level keys: ['marketid', 'symbol', 'timestamp', 'sequence', 'bids', 'asks']
  levels returned: 75 bids / 81 asks
  raw bid[0]: {"price": "34.63", "quantity": "0.009869"}
  raw ask[0]: {"price": "34.99", "quantity": "0.008954"}
  best bid 34.63 / best ask 34.99 / mid 34.810000 / spread 1.034%
  TOTAL book notional: bids $15,100  asks $18,409
  +- 2.0% band: $        2.40   (2 bids, 2 asks in band)
  +- 5.0% band: $   29,820.48   (36 bids, 22 asks in band)
  +-10.0% band: $   29,899.77   (44 bids, 27 asks in band)
  +-25.0% band: $   30,171.71   (66 bids, 42 asks in band)
  nearest 6 bids: [(34.63, 0.009869), (34.24, 0.025766), (33.97, 0.025409), (33.92, 14.21742), (33.91, 14.726911), (33.9, 16.931647)]
  nearest 6 asks: [(34.99, 0.008954), (35.3, 0.024384), (35.61, 0.024065), (35.7, 15.24132), (35.71, 15.542611), (35.72, 15.871736)]
XMR/USDT
  HTTP 200
  top-level keys: ['marketid', 'symbol', 'timestamp', 'sequence', 'bids', 'asks']
  levels returned: 100 bids